# SmartFit LoRA overfit test: IDM-VTON (spec §8.4)

Trains a LoRA adapter on **8 samples** for 300 steps, then generates those 8 samples with the adapter off
(the pretrained IDM-VTON) and on. It answers two questions before any long training run:

1. **Does LoRA training fit** on Kaggle's GPUs? (Peak memory per GPU is logged.)
2. **Does the adapter learn?** On its own 8 training samples the LoRA output must be clearly closer to the
   ground truth (lower LPIPS) than the base model's.

**Before you click Run All** (right-hand panel → Session options):
1. **Accelerator:** **GPU T4 x2** (needed: the try-on UNet trains on one T4, the frozen parts sit on the other).
2. **Internet:** On (code, packages, ~28 GB of model weights on first run).
3. **Input → Add Input:** your private *SmartFit LoRA overfit pack* dataset.
4. Optional **Add-ons → Secrets:** `HF_TOKEN` (Hugging Face read token).

Expected run time: about 1 hour (≈10 min downloads and conversion, ≈20–30 min training, ≈15 min evaluation).
Safe to re-run in the same session: the checkout, the env and the weights in `/tmp` are reused.

In [ ]:
import glob, json, os

hits = sorted(glob.glob("/kaggle/input/**/code/train_lora.py", recursive=True))
assert hits, "LoRA pack not found: add the SmartFit LoRA overfit pack dataset as an input."
PACK = os.path.dirname(os.path.dirname(hits[0]))
DATA, MANIFEST = f"{PACK}/data", f"{PACK}/manifests/overfit.jsonl"
print("pack:", PACK)
print(open(f"{PACK}/overfit_ids.txt").read())
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("No HF_TOKEN secret: anonymous downloads (fine for this public model)")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code at a fixed commit and an isolated environment

In [ ]:
![ -d /tmp/IDM-VTON ] || git clone -q https://github.com/yisol/IDM-VTON.git /tmp/IDM-VTON
!git -C /tmp/IDM-VTON checkout -q 0d5f3ec2d737487a9bb24e4100936ad254780383 && git -C /tmp/IDM-VTON log -1 --format="%H %cd"
!pip install -q uv==0.5.11
![ -x /tmp/env/bin/python ] || uv venv -q --python 3.11 /tmp/env
!uv pip install -q --python /tmp/env/bin/python torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /tmp/env/bin/python -r {PACK}/code/requirements-lora.txt
!/tmp/env/bin/python -c "import torch, peft; print(torch.__version__, 'peft', peft.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

## 2. Train: 300 steps on the 8 samples
A line every 5 steps with the loss and the peak memory per GPU.

In [ ]:
!/tmp/env/bin/python {PACK}/code/train_lora.py --repo /tmp/IDM-VTON --manifest {MANIFEST} --data-root {DATA} --out /kaggle/working/lora_overfit --overfit 8

In [ ]:
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open("/kaggle/working/lora_overfit/metrics.jsonl")]
run = json.load(open("/kaggle/working/lora_overfit/run.json"))
print(json.dumps({k: run.get(k) for k in ("status", "final_step", "minutes", "peak_reserved_gb_by_gpu",
                                          "trainable_parameters")}, indent=2))
print("seconds per step (median):", sorted(r["sec_per_step"] for r in rows)[len(rows) // 2])
plt.figure(figsize=(7, 3))
plt.plot([r["step"] for r in rows], [r["loss"] for r in rows])
plt.xlabel("step"); plt.ylabel("loss (mean of 5 steps)"); plt.title("LoRA overfit loss"); plt.grid(alpha=.3)
plt.show()

## 3. Base vs LoRA on the same 8 samples
Same seed and inputs; only the adapter is switched off/on.

In [ ]:
!/tmp/env/bin/python {PACK}/code/eval_lora.py --repo /tmp/IDM-VTON --manifest {MANIFEST} --data-root {DATA} --adapter /kaggle/working/lora_overfit/adapter --out /kaggle/working/lora_overfit/eval

In [ ]:
from IPython.display import Image as Show, display

ev = json.load(open("/kaggle/working/lora_overfit/eval/results.json"))
print(json.dumps(ev["summary"], indent=2))
display(Show(filename="/kaggle/working/lora_overfit/eval/sheet.jpg", width=900))

## 4. Download
Download `results_lora_overfit.zip` from the **Output** panel and add it to the repo's `backend/` folder as before. It includes the adapter, metrics, run summary and the evaluation images (checkpoints are left out).

In [ ]:
!cd /kaggle/working && rm -f results_lora_overfit.zip && zip -qr results_lora_overfit.zip lora_overfit -x "lora_overfit/checkpoints/*" && ls -la /kaggle/working